Tree Species Data 

In [1]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns

# Load trees
trees = gpd.read_file("../data/raw/trees-with-species-and-dimensions-urban-forest.geojson")

print(f"Trees loaded: {trees.shape}")

Trees loaded: (82064, 20)


In [2]:
# Step 1: Drop unnecessary columns
trees = trees.drop(columns=['uploaddate', 'easting', 'northing', 'geolocation'])

# Step 2: Check DBH outliers before fixing
print("DBH before cleaning:")
print(f"  Trees with DBH > 200cm: {(trees['diameter_breast_height'] > 200).sum()}")
print(f"  Trees with DBH > 100cm: {(trees['diameter_breast_height'] > 100).sum()}")
print(f"  Missing DBH: {trees['diameter_breast_height'].isnull().sum()}")

# Cap DBH at 200cm 
trees.loc[trees['diameter_breast_height'] > 200, 'diameter_breast_height'] = None

print(f"\nDBH after capping outliers:")
print(trees['diameter_breast_height'].describe())

DBH before cleaning:
  Trees with DBH > 200cm: 54
  Trees with DBH > 100cm: 482
  Missing DBH: 44863

DBH after capping outliers:
count    37147.000000
mean        34.354968
std         23.819303
min          1.000000
25%         18.000000
50%         30.000000
75%         45.000000
max        200.000000
Name: diameter_breast_height, dtype: float64


In [3]:
# Step 3: Fill missing DBH with species median
trees['diameter_breast_height'] = trees.groupby('common_name')['diameter_breast_height'].transform(
    lambda x: x.fillna(x.median())
)

# Check if any are still missing (species where ALL trees had missing DBH)
remaining_missing = trees['diameter_breast_height'].isnull().sum()
print(f"Still missing after species median fill: {remaining_missing}")

# Fill any remaining with the overall median
if remaining_missing > 0:
    overall_median = trees['diameter_breast_height'].median()
    trees['diameter_breast_height'] = trees['diameter_breast_height'].fillna(overall_median)
    print(f"Filled remaining with overall median: {overall_median}")

print(f"\nFinal DBH stats:")
print(trees['diameter_breast_height'].describe())
print(f"\nMissing DBH: {trees['diameter_breast_height'].isnull().sum()}")

Still missing after species median fill: 1946
Filled remaining with overall median: 21.5

Final DBH stats:
count    82064.000000
mean        29.084885
std         18.490648
min          1.000000
25%         20.000000
50%         21.500000
75%         35.000000
max        200.000000
Name: diameter_breast_height, dtype: float64

Missing DBH: 0


/Users/aidanpage/Documents/GitHub/MOP-Code/Playground/AidanPage_T126/venv/lib/python3.9/site-packages/numpy/lib/_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/aidanpage/Documents/GitHub/MOP-Code/Playground/AidanPage_T126/venv/lib/python3.9/site-packages/numpy/lib/_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/aidanpage/Documents/GitHub/MOP-Code/Playground/AidanPage_T126/venv/lib/python3.9/site-packages/numpy/lib/_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/aidanpage/Documents/GitHub/MOP-Code/Playground/AidanPage_T126/venv/lib/python3.9/site-packages/numpy/lib/_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/aidanpage/Documents/GitHub/MOP-Code/Playground/AidanPage_T126

In [4]:
# Step 4: Create risk target variable from useful life expectancy
trees['risk_class'] = trees['useful_life_expectency_value'].map({
    10: 'HIGH',
    20: 'HIGH',
    30: 'MEDIUM',
    40: 'LOW',
    50: 'LOW'
})

print("Risk class distribution:")
print(trees['risk_class'].value_counts())
print(f"\nAs percentages:")
print(trees['risk_class'].value_counts(normalize=True).round(3) * 100)

# Step 5: Calculate tree age
trees['year_planted'] = pd.to_numeric(trees['year_planted'], errors='coerce')
trees['tree_age'] = 2026 - trees['year_planted']

print(f"\nTree age stats:")
print(trees['tree_age'].describe())

Risk class distribution:
risk_class
LOW       56003
MEDIUM    21115
HIGH       4946
Name: count, dtype: int64

As percentages:
risk_class
LOW       68.2
MEDIUM    25.7
HIGH       6.0
Name: proportion, dtype: float64

Tree age stats:
count    82064.000000
mean        47.688816
std         54.307899
min          1.000000
25%          8.000000
50%         14.000000
75%        126.000000
max        126.000000
Name: tree_age, dtype: float64


In [6]:
# Step 6: Save cleaned trees
trees.to_file("../data/processed/trees_cleaned.geojson", driver="GeoJSON")
print(f"Cleaned trees saved: {trees.shape}")
print(f"Columns: {trees.columns.tolist()}")

Cleaned trees saved: (82064, 18)
Columns: ['com_id', 'common_name', 'scientific_name', 'genus', 'family', 'diameter_breast_height', 'year_planted', 'date_planted', 'age_description', 'useful_life_expectency', 'useful_life_expectency_value', 'precinct', 'located_in', 'latitude', 'longitude', 'geometry', 'risk_class', 'tree_age']


Microclimate Sensor data cleaning

In [8]:
# Load and clean microclimate sensors
sensors = pd.read_csv("../data/raw/microclimate-sensors-data.csv")

# Parse timestamp with UTC to handle mixed timezones
sensors['Time'] = pd.to_datetime(sensors['Time'], utc=True)

# Split LatLong into separate columns
sensors[['lat', 'lon']] = sensors['LatLong'].str.split(',', expand=True).astype(float)

# Drop rows with no coordinates
sensors = sensors.dropna(subset=['lat', 'lon'])

# Extract date for daily aggregation
sensors['date'] = sensors['Time'].dt.date

print(f"Shape after cleaning: {sensors.shape}")
print(f"\nSample of parsed data:")
sensors[['Time', 'date', 'lat', 'lon', 'AirTemperature', 'RelativeHumidity']].head()

Shape after cleaning: (580320, 19)

Sample of parsed data:


,Time,date,lat,lon,AirTemperature,RelativeHumidity
0,2025-11-16 02:17:20+00:00,2025-11-16,-37.822331,144.952170,15.6,71.6
1,2025-11-15 16:17:34+00:00,2025-11-15,-37.819499,144.978721,15.2,78.9
2,2025-11-16 02:14:29+00:00,2025-11-16,-37.814035,144.967280,15.7,79.0
3,2025-08-17 15:10:39+00:00,2025-08-17,-37.814035,144.967280,9.1,71.8
4,2025-11-15 16:23:06+00:00,2025-11-15,-37.818593,144.971640,14.6,89.0


In [9]:
# Aggregate to daily averages per sensor
sensor_daily = sensors.groupby(['SensorLocation', 'lat', 'lon', 'date']).agg({
    'AirTemperature': 'mean',
    'RelativeHumidity': 'mean'
}).reset_index()

sensor_daily.columns = ['sensor_location', 'lat', 'lon', 'date', 'avg_temp', 'avg_humidity']

print(f"Daily sensor data shape: {sensor_daily.shape}")
print(f"\nSample:")
sensor_daily.head()

# Save
sensor_daily.to_csv("../data/processed/sensors_daily.csv", index=False)
print(f"\nSaved to data/processed/sensors_daily.csv")

Daily sensor data shape: (6443, 6)

Sample:

Saved to data/processed/sensors_daily.csv


BOM weather data cleaning

In [10]:
# Load BOM data
bom_temp = pd.read_csv("../data/raw/IDCJAC0010_086338_1800_Data.csv")
bom_rain = pd.read_csv("../data/raw/IDCJAC0009_086338_1800_Data.csv")

# Create date column for both
bom_temp['date'] = pd.to_datetime(bom_temp[['Year', 'Month', 'Day']])
bom_rain['date'] = pd.to_datetime(bom_rain[['Year', 'Month', 'Day']])

# Keep only whats needed
bom_temp = bom_temp[['date', 'Maximum temperature (Degree C)']].rename(
    columns={'Maximum temperature (Degree C)': 'max_temp'}
)
bom_rain = bom_rain[['date', 'Rainfall amount (millimetres)']].rename(
    columns={'Rainfall amount (millimetres)': 'rainfall_mm'}
)

# Merge on date
weather = bom_temp.merge(bom_rain, on='date', how='outer')

# Drop rows where both are missing
weather = weather.dropna(subset=['max_temp', 'rainfall_mm'], how='all')

# Fill missing rainfall with 0 (no recorded rain = no rain)
weather['rainfall_mm'] = weather['rainfall_mm'].fillna(0)

# Interpolate short temp gaps
weather = weather.sort_values('date')
weather['max_temp'] = weather['max_temp'].interpolate(method='linear', limit=3)

print(f"Weather shape: {weather.shape}")
print(f"\nMissing values:\n{weather.isnull().sum()}")
print(f"\nDate range: {weather['date'].min()} to {weather['date'].max()}")
print(f"\nSample:")
weather.head()

Weather shape: (4679, 3)

Missing values:
date           0
max_temp       0
rainfall_mm    0
dtype: int64

Date range: 2013-06-01 00:00:00 to 2026-03-24 00:00:00

Sample:


,date,max_temp,rainfall_mm
151,2013-06-01,15.8,0.0
152,2013-06-02,15.7,5.0
153,2013-06-03,14.8,0.2
154,2013-06-04,15.0,0.2
155,2013-06-05,14.6,0.0


In [11]:
# Save weather
weather.to_csv("../data/processed/weather_cleaned.csv", index=False)
print("Weather saved.")

Weather saved.


Soil Sensor Data Cleaning 

In [12]:
# Clean soil sensor data — load only moisture 
soil = pd.read_csv("../data/raw/soil-sensor-readings-historical-data.csv")
soil_locations = pd.read_csv("../data/raw/soil-sensor-locations.csv")

# Filter to just soil moisture readings
soil_moisture = soil[soil['Unit'] == '%VWC'].copy()

# Parse timestamp
soil_moisture['Local_Time'] = pd.to_datetime(soil_moisture['Local_Time'], utc=True)
soil_moisture['date'] = soil_moisture['Local_Time'].dt.date

# Aggregate to daily average moisture per site
soil_daily = soil_moisture.groupby(['Site_ID', 'date']).agg({
    'Soil_Value': 'mean'
}).reset_index()
soil_daily.columns = ['site_id', 'date', 'avg_soil_moisture']

# Join to locations for coordinates
soil_daily = soil_daily.merge(
    soil_locations[['Site_ID', 'Latitude', 'Longitude']],
    left_on='site_id',
    right_on='Site_ID',
    how='left'
).drop(columns=['Site_ID'])

print(f"Daily soil moisture shape: {soil_daily.shape}")
print(f"\nMissing values:\n{soil_daily.isnull().sum()}")
print(f"\nSample:")
soil_daily.head()

Daily soil moisture shape: (33115, 5)

Missing values:
site_id              0
date                 0
avg_soil_moisture    0
Latitude             0
Longitude            0
dtype: int64

Sample:


,site_id,date,avg_soil_moisture,Latitude,Longitude
0,64970,2023-09-02,33.529375,-37.7864,144.96259
1,64970,2023-09-03,33.350313,-37.7864,144.96259
2,64970,2023-09-04,33.352187,-37.7864,144.96259
3,64970,2023-09-05,33.262500,-37.7864,144.96259
4,64970,2023-09-06,33.066875,-37.7864,144.96259


In [13]:
soil_daily.to_csv("../data/processed/soil_daily.csv", index=False)
print("Soil data saved.")

Soil data saved.


Coordinate Reference System (CRS)

In [14]:
# Check current CRS of trees
print(f"Trees CRS: {trees.crs}")

# Set project CRS — GDA2020 (EPSG:7844) for Melbourne
trees = trees.to_crs("EPSG:7844")
print(f"Trees reprojected to: {trees.crs}")

# Convert sensor data to GeoDataFrame and set CRS
sensor_daily = pd.read_csv("../data/processed/sensors_daily.csv")
sensor_gdf = gpd.GeoDataFrame(
    sensor_daily,
    geometry=gpd.points_from_xy(sensor_daily['lon'], sensor_daily['lat']),
    crs="EPSG:4326"  # Raw GPS coordinates are WGS84
).to_crs("EPSG:7844")
print(f"Sensors reprojected to: {sensor_gdf.crs}")

# Convert soil data to GeoDataFrame and set CRS
soil_daily = pd.read_csv("../data/processed/soil_daily.csv")
soil_gdf = gpd.GeoDataFrame(
    soil_daily,
    geometry=gpd.points_from_xy(soil_daily['Longitude'], soil_daily['Latitude']),
    crs="EPSG:4326"
).to_crs("EPSG:7844")
print(f"Soil reprojected to: {soil_gdf.crs}")

Trees CRS: EPSG:4326
Trees reprojected to: EPSG:7844
Sensors reprojected to: EPSG:7844
Soil reprojected to: EPSG:7844
